In [1]:
AGENTS = [
    "20250603_Refact_Agent_claude-4-sonnet",
    "20250720_Lingxi-v1.5_claude-4-sonnet-20250514",
    "20250805_openhands-Qwen3-Coder-480B-A35B-Instruct",
    "20250928_trae_doubao_seed_code",
    "20250807_mini-v1.7.0_gpt-5-mini",
]

In [2]:
import json
import os
import pandas as pd
import numpy as np
import pprint as pp
from scipy.stats import wilcoxon
from dataset.extract_ground_truths.effect.process_agent_patch import get_diff_info_per_instance
from execution.util import get_instance_ids

/home/yusuf/explainbench/explainbench/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
resolved_dict = {}
base_path = "/home/yusuf/explainbench/shared_logs/logs/run_evaluation/output_per_step/efficacy"
for agent in AGENTS:
    path = agent + ".json" if "mini-" not in agent else agent + "_resolved.json"
    path = os.path.join(base_path, path)
    with open(path, "r") as f:
        temp = json.load(f)
    resolved_dict[agent] = temp["resolved"]

# End-to-End

In [4]:
ee_intent = "results_intent_ee/final_results_intent_pbtassertionmcq.json"
with open(ee_intent, "r") as f:
    ee_intent = json.load(f)

In [5]:
def extract_score_local_ee(input_dict, agent, q_type, resolved_dict):
    scores_dict = input_dict[agent]
    instance_ids = []
    agents = []
    scores = []
    q_types = []
    trials = []
    is_resolved = []
    answers = []
    gts = []
    choices = []
    for id_, result_dict in scores_dict.items():
        temp_scores = result_dict["individual_scores"]
        for idx, s in enumerate(temp_scores):
            scores.append(s)
            trials.append(idx+1)
            instance_ids.append(id_)
            is_resolved.append(id_ in resolved_dict[agent])
            agents.append(agent)
            q_types.append(q_type)
            answers.append(result_dict["all_pred"][idx]["selection"])
            gts.append(result_dict["answer_gt"])
            choices.append(result_dict["choices"])
    return agents, instance_ids, scores, q_types, is_resolved, trials, answers, gts, choices 

In [6]:
instance_ids = []
scores = []
agents = []
q_types = []
trials = []
resolved = []
answers = []
gts = []
choices = []
for agent in (AGENTS):
    temp_agents, temp_instance_ids, temp_scores, temp_q_types, temp_resolved, temp_trials, temp_answers, temp_gts, temp_choices = extract_score_local_ee(ee_intent, agent, "ee_intent", resolved_dict)
    agents.extend(temp_agents)
    q_types.extend(temp_q_types)
    instance_ids.extend(temp_instance_ids)
    scores.extend(temp_scores)
    resolved.extend(temp_resolved)
    trials.extend(temp_trials)
    answers.extend(temp_answers)
    gts.extend(temp_gts)
    choices.extend(temp_choices)
df_dict = {
    "agent_name": agents,
    "q_type": q_types,
    "trial": trials,
    "id_": instance_ids,
    "answer": answers,
    "choices": choices,
    "score": scores,
    "resolved": resolved,
    "truth": gts
}        

ee_intent = pd.DataFrame(df_dict)

In [7]:
answer_to_idx = {
    "A": 0,
    "B": 1,
    "C": 2,
    "D": 3,
    "E": 4
}
def is_informative(answer, choices):
    idx = answer_to_idx[answer]
    selected = choices[idx]
    if selected == "The question cannot be answered based on the explanation alone.":
        return False
    return True

In [8]:
ee_intent["informative"] = ee_intent.apply(lambda x: is_informative(x.answer, x.choices), axis=1)
ee_intent.informative.value_counts()

informative
True     5126
False    2299
Name: count, dtype: int64

In [35]:
ee_intent["label"] = ee_intent.apply(lambda x: "not-informative" if not x.informative else "null", axis=1)
ee_intent["label"] = ee_intent.apply(lambda x: "misaligned" if x.informative and np.allclose(x.score, 0.0) else x.label, axis=1)
ee_intent["label"] = ee_intent.apply(lambda x: "aligned" if x.informative and np.allclose(x.score, 1.0) else x.label, axis=1)

In [36]:
ee_intent.head(3)

,agent_name,q_type,trial,id_,answer,choices,score,resolved,truth,informative,label
0,20250603_Refact_Agent_claude-4-sonnet,ee_intent,1,astropy__astropy-13977,A,[The question cannot be answered based on the ...,False,False,B,False,not-informative
1,20250603_Refact_Agent_claude-4-sonnet,ee_intent,2,astropy__astropy-13977,A,[The question cannot be answered based on the ...,False,False,B,False,not-informative
2,20250603_Refact_Agent_claude-4-sonnet,ee_intent,3,astropy__astropy-13977,A,[The question cannot be answered based on the ...,False,False,B,False,not-informative


In [37]:
ee_effect = "results_effect_ee/final_results_effect_pbtresultmcq.json"
with open(ee_effect, "r") as f:
    ee_effect = json.load(f)

In [38]:
def extract_score_local_ee(input_dict, agent, q_type, resolved_dict):
    scores_dict = input_dict[agent]
    instance_ids = []
    agents = []
    scores = []
    q_types = []
    trials = []
    is_resolved = []
    answers = []
    gts = []
    choices = []
    for id_, result_dict in scores_dict.items():
        temp_scores = result_dict["individual_scores"]
        for idx, s in enumerate(temp_scores):
            scores.append(s)
            trials.append(idx+1)
            instance_ids.append(id_)
            is_resolved.append(id_ in resolved_dict[agent])
            agents.append(agent)
            q_types.append(q_type)
            choices.append(result_dict["choices"])
            answers.append([result_dict["all_pred"][idx]["before_selection"], result_dict["all_pred"][idx]["after_selection"]])
            gts.append(result_dict["answer_gt"])
    return agents, instance_ids, scores, q_types, is_resolved, trials, answers, gts, choices 

In [39]:
instance_ids = []
scores = []
agents = []
q_types = []
trials = []
resolved = []
answers = []
gts = []
choices = []
for agent in (AGENTS):
    temp_agents, temp_instance_ids, temp_scores, temp_q_types, temp_resolved, temp_trials, temp_answers, temp_gts, temp_choices = extract_score_local_ee(ee_effect, agent, "ee_effect", resolved_dict)
    agents.extend(temp_agents)
    q_types.extend(temp_q_types)
    instance_ids.extend(temp_instance_ids)
    scores.extend(temp_scores)
    resolved.extend(temp_resolved)
    trials.extend(temp_trials)
    answers.extend(temp_answers)
    gts.extend(temp_gts)
    choices.extend(temp_choices)
df_dict = {
    "agent_name": agents,
    "q_type": q_types,
    "trial": trials,
    "id_": instance_ids,
    "answer": answers,
    "score": scores,
    "resolved": resolved,
    "truth": gts,
    "choices": choices
}        

ee_effect = pd.DataFrame(df_dict)

In [40]:
answer_to_idx = {
    "A": 0,
    "B": 1,
    "C": 2,
    "D": 3,
    "E": 4
}
def is_expl_enough_before(truths, choices):
    # before selected answer
    idx = answer_to_idx[truths[0][0]]
    selected = choices[idx]
    if selected == "The question cannot be answered based on the explanation alone.":
        return False
    return True

In [41]:
ee_effect["expl_enough_before"] = ee_effect.apply(lambda x: is_expl_enough_before(x.answer, x.choices), axis=1)

In [42]:
answer_to_idx = {
    "A": 0,
    "B": 1,
    "C": 2,
    "D": 3,
    "E": 4
}
def is_expl_enough_after(truths, choices):
    # before selected answer
    idx = answer_to_idx[truths[1][0]]
    selected = choices[idx]
    if selected == "The question cannot be answered based on the explanation alone.":
        return False
    return True

In [43]:
ee_effect["expl_enough_after"] = ee_effect.apply(lambda x: is_expl_enough_after(x.answer, x.choices), axis=1)

In [44]:
ee_effect["informative"] = ee_effect.apply(lambda x: x.expl_enough_before == True and x.expl_enough_after == True, axis=1)

In [45]:
ee_effect["label"] = ee_effect.informative.apply(lambda x: "not-informative" if not x else "null")
ee_effect["label"] = ee_effect.apply(lambda x: "misaligned" if x.informative and np.allclose(x.score, 0.0) else x.label, axis=1)
ee_effect["label"] = ee_effect.apply(lambda x: "aligned" if x.informative and np.allclose(x.score, 1.0) else x.label, axis=1)

In [46]:
ee_intent.head(3)

,agent_name,q_type,trial,id_,answer,choices,score,resolved,truth,informative,label
0,20250603_Refact_Agent_claude-4-sonnet,ee_intent,1,astropy__astropy-13977,A,[The question cannot be answered based on the ...,False,False,B,False,not-informative
1,20250603_Refact_Agent_claude-4-sonnet,ee_intent,2,astropy__astropy-13977,A,[The question cannot be answered based on the ...,False,False,B,False,not-informative
2,20250603_Refact_Agent_claude-4-sonnet,ee_intent,3,astropy__astropy-13977,A,[The question cannot be answered based on the ...,False,False,B,False,not-informative


In [47]:
ee_effect.drop(columns=["expl_enough_before", "expl_enough_after"], inplace=True)

In [48]:
ee_effect.head(3)

,agent_name,q_type,trial,id_,answer,score,resolved,truth,choices,informative,label
0,20250603_Refact_Agent_claude-4-sonnet,ee_effect,1,django__django-12304,"[A, A]",False,True,"[D, E]",[The question cannot be answered based on the ...,False,not-informative
1,20250603_Refact_Agent_claude-4-sonnet,ee_effect,2,django__django-12304,"[A, A]",False,True,"[D, E]",[The question cannot be answered based on the ...,False,not-informative
2,20250603_Refact_Agent_claude-4-sonnet,ee_effect,3,django__django-12304,"[A, A]",False,True,"[D, E]",[The question cannot be answered based on the ...,False,not-informative


In [49]:
ee = pd.concat([ee_effect, ee_intent])

In [50]:
label_mean = (
    ee.groupby(["agent_name", "q_type"])["label"]
          .value_counts(normalize=True)
          .rename("percentage")
          .reset_index()
)

label_mean = label_mean[label_mean.label != "aligned"]
label_mean.sort_values(by=["agent_name", "q_type", "label"])

,agent_name,q_type,label,percentage
1,20250603_Refact_Agent_claude-4-sonnet,ee_effect,misaligned,0.237710
2,20250603_Refact_Agent_claude-4-sonnet,ee_effect,not-informative,0.048485
5,20250603_Refact_Agent_claude-4-sonnet,ee_intent,misaligned,0.040404
4,20250603_Refact_Agent_claude-4-sonnet,ee_intent,not-informative,0.237037
7,20250720_Lingxi-v1.5_claude-4-sonnet-20250514,ee_effect,misaligned,0.249158
8,20250720_Lingxi-v1.5_claude-4-sonnet-20250514,ee_effect,not-informative,0.035690
11,20250720_Lingxi-v1.5_claude-4-sonnet-20250514,ee_intent,misaligned,0.045791
10,20250720_Lingxi-v1.5_claude-4-sonnet-20250514,ee_intent,not-informative,0.249832
13,20250805_openhands-Qwen3-Coder-480B-A35B-Instruct,ee_effect,misaligned,0.276768
14,20250805_openhands-Qwen3-Coder-480B-A35B-Instruct,ee_effect,not-informative,0.025589


In [51]:
label_mean[label_mean.q_type =="ee_effect"]

,agent_name,q_type,label,percentage
1,20250603_Refact_Agent_claude-4-sonnet,ee_effect,misaligned,0.237710
2,20250603_Refact_Agent_claude-4-sonnet,ee_effect,not-informative,0.048485
7,20250720_Lingxi-v1.5_claude-4-sonnet-20250514,ee_effect,misaligned,0.249158
8,20250720_Lingxi-v1.5_claude-4-sonnet-20250514,ee_effect,not-informative,0.035690
13,20250805_openhands-Qwen3-Coder-480B-A35B-Instruct,ee_effect,misaligned,0.276768
14,20250805_openhands-Qwen3-Coder-480B-A35B-Instruct,ee_effect,not-informative,0.025589
19,20250807_mini-v1.7.0_gpt-5-mini,ee_effect,misaligned,0.335354
20,20250807_mini-v1.7.0_gpt-5-mini,ee_effect,not-informative,0.156902
25,20250928_trae_doubao_seed_code,ee_effect,misaligned,0.239057
26,20250928_trae_doubao_seed_code,ee_effect,not-informative,0.044444


We further analyze the end-to-end effects of a patch by evaluating whether an LLM can correctly infer the patch’s outcome solely from the agent-provided explanation. An explanation is considered misaligned if it contains information about the patch’s effect but leads the LLM evaluator to an incorrect conclusion (i.e., the information is misleading or erroneous). In contrast, an explanation is considered non-informative if it does not provide sufficient information to determine the end-to-end effect of the patch at all.

- Across all agents, the percentage of misaligned explanations is substantially higher than not-informative explanations.
- This strongly suggests that failures in end-to-end explanations are primarily due to misleading or incorrect information, not missing information. In other words:
- The agent 20250807_mini-v1.7.0_gpt-5-mini stands out sharply. This agent appears to:
    - Frequently fail to describe the end-to-end effect at all, and
    - When it does attempt an explanation, it is often incorrect.

In [52]:
label_mean[label_mean.q_type =="ee_intent"]

,agent_name,q_type,label,percentage
4,20250603_Refact_Agent_claude-4-sonnet,ee_intent,not-informative,0.237037
5,20250603_Refact_Agent_claude-4-sonnet,ee_intent,misaligned,0.040404
10,20250720_Lingxi-v1.5_claude-4-sonnet-20250514,ee_intent,not-informative,0.249832
11,20250720_Lingxi-v1.5_claude-4-sonnet-20250514,ee_intent,misaligned,0.045791
16,20250805_openhands-Qwen3-Coder-480B-A35B-Instruct,ee_intent,not-informative,0.245118
17,20250805_openhands-Qwen3-Coder-480B-A35B-Instruct,ee_intent,misaligned,0.032323
21,20250807_mini-v1.7.0_gpt-5-mini,ee_intent,not-informative,0.498316
23,20250807_mini-v1.7.0_gpt-5-mini,ee_intent,misaligned,0.034343
28,20250928_trae_doubao_seed_code,ee_intent,not-informative,0.317845
29,20250928_trae_doubao_seed_code,ee_intent,misaligned,0.045791


For intent-focused questions, explanation failures are dominated by non-informativeness rather than misalignment. Across all agents, a substantial fraction of explanations omit intent-relevant information entirely, while incorrect intent statements remain rare.

# Local

In [90]:
def extract_score_local_intent(input_dict, agent, q_type, resolved_dict):
    scores_dict = input_dict[agent]
    instance_ids = []
    agents = []
    scores = []
    q_types = []
    trials = []
    is_resolved = []
    answers = []
    for id_, result_dict in scores_dict.items():
        temp_scores = result_dict["individual_scores"]
        for idx, s in enumerate(temp_scores):
            scores.append(s)
            trials.append(idx+1)
            instance_ids.append(id_)
            is_resolved.append(id_ in resolved_dict[agent])
            agents.append(agent)
            q_types.append(q_type)
            answers.append(result_dict["all_pred"][idx][0])
    return agents, instance_ids, scores, q_types, is_resolved, trials, answers

In [91]:
local_intent = "results_intent_local/eval.individual.intent.json"
with open(local_intent, "r") as f:
    local_intent = json.load(f)

In [92]:
instance_ids = []
scores = []
agents = []
q_types = []
trials = []
resolved = []
answers = []
for agent in (AGENTS):
    temp_agents, temp_instance_ids, temp_scores, temp_q_types, temp_resolved, temp_trials, temp_answers = extract_score_local_intent(local_intent, agent, "local_intent", resolved_dict)
    agents.extend(temp_agents)
    q_types.extend(temp_q_types)
    instance_ids.extend(temp_instance_ids)
    scores.extend(temp_scores)
    resolved.extend(temp_resolved)
    trials.extend(temp_trials)
    answers.extend(temp_answers)
df_dict = {
    "agent_name": agents,
    "q_type": q_types,
    "trial": trials,
    "id_": instance_ids,
    "answer": answers,
    "score": scores,
    "resolved": resolved
}        

local_intent = pd.DataFrame(df_dict)

In [93]:
ANSWER_JSON = "/home/yusuf/explainbench/shared_logs/logs/run_evaluation/output_per_step/experiment_w_reachability/step4.intent.json"
with open(ANSWER_JSON, "r") as f:
    answer_json = json.load(f)

answers = []
for idx, row in local_intent.iterrows():
    agent = row["agent_name"]
    id_ = row["id_"]
    answers.append(
        answer_json[agent][id_]["answer"][0])
    
local_intent["truth"] = answers

In [94]:
local_intent["answer"] =local_intent.answer.str.upper()
local_intent["truth"] =local_intent.truth.str.upper()

In [95]:
local_intent["informative"] = local_intent.apply(lambda x: x.answer != "E", axis=1)

In [96]:
local_intent.truth.value_counts()

truth
A    1925
D    1900
C    1900
B    1700
Name: count, dtype: int64

In [97]:
local_intent.answer.value_counts()

answer
E    2267
D    1471
C    1371
A    1197
B    1119
Name: count, dtype: int64

In [98]:
local_intent.informative.value_counts()

informative
True     5158
False    2267
Name: count, dtype: int64

In [99]:
local_intent.head(3)

,agent_name,q_type,trial,id_,answer,score,resolved,truth,informative
0,20250603_Refact_Agent_claude-4-sonnet,local_intent,1,django__django-11179,D,1.0,True,D,True
1,20250603_Refact_Agent_claude-4-sonnet,local_intent,2,django__django-11179,D,1.0,True,D,True
2,20250603_Refact_Agent_claude-4-sonnet,local_intent,3,django__django-11179,D,1.0,True,D,True


In [100]:
local_intent["label"] = "not-informative"
local_intent["label"] = local_intent.apply(lambda x: "misaligned" if x.informative and np.allclose(x.score, 0.0) else x.label, axis=1)
local_intent["label"] = local_intent.apply(lambda x: "aligned" if x.informative and np.allclose(x.score, 1.0) else x.label, axis=1)

In [102]:
local_intent.truth.value_counts()

truth
A    1925
D    1900
C    1900
B    1700
Name: count, dtype: int64

In [103]:
label_mean = (
    local_intent.groupby(["agent_name"])["label"]
          .value_counts(normalize=True)
          .rename("mean")
          .reset_index()
)

label_mean = label_mean[label_mean.label != "aligned"]

In [104]:
label_mean.sort_values(by=["agent_name", "label"]).round(3)

,agent_name,label,mean
0,20250603_Refact_Agent_claude-4-sonnet,misaligned,0.374
2,20250603_Refact_Agent_claude-4-sonnet,not-informative,0.271
4,20250720_Lingxi-v1.5_claude-4-sonnet-20250514,misaligned,0.362
5,20250720_Lingxi-v1.5_claude-4-sonnet-20250514,not-informative,0.269
6,20250805_openhands-Qwen3-Coder-480B-A35B-Instruct,misaligned,0.355
8,20250805_openhands-Qwen3-Coder-480B-A35B-Instruct,not-informative,0.292
11,20250807_mini-v1.7.0_gpt-5-mini,misaligned,0.299
9,20250807_mini-v1.7.0_gpt-5-mini,not-informative,0.399
12,20250928_trae_doubao_seed_code,misaligned,0.373
14,20250928_trae_doubao_seed_code,not-informative,0.295


In [105]:
local_effect = "results_effect_local/eval.individual.effect.json"

In [106]:
with open(local_effect, "r") as f:
    local_effect = json.load(f)

In [107]:
instance_ids = []
scores = []
agents = []
q_types = []
trials = []
resolved = []
answers = []
for agent in (AGENTS):
    temp_agents, temp_instance_ids, temp_scores, temp_q_types, temp_resolved, temp_trials, temp_answers = extract_score_local_intent(local_effect, agent, "local_effect", resolved_dict)
    agents.extend(temp_agents)
    q_types.extend(temp_q_types)
    instance_ids.extend(temp_instance_ids)
    scores.extend(temp_scores)
    resolved.extend(temp_resolved)
    trials.extend(temp_trials)
    answers.extend(temp_answers)
df_dict = {
    "agent_name": agents,
    "q_type": q_types,
    "trial": trials,
    "id_": instance_ids,
    "answer": answers,
    "score": scores,
    "resolved": resolved
}        

local_effect = pd.DataFrame(df_dict)

In [108]:
ANSWER_JSON = "/home/yusuf/explainbench/shared_logs/logs/run_evaluation/output_per_step/experiment_w_reachability/step4.json"
with open(ANSWER_JSON, "r") as f:
    answer_json = json.load(f)


answers = []
for idx, row in local_effect.iterrows():
    agent = row["agent_name"]
    id_ = row["id_"]
    answers.append(
        answer_json[agent][id_]["answer"][0])
    
local_effect["truth"] = answers

In [110]:
local_effect["answer"] = local_effect.answer.str.upper()
local_effect["truth"] = local_effect.truth.str.upper()

In [112]:
local_effect["informative"] = local_effect.apply(lambda x: x.answer != "F", axis=1)

In [113]:
local_effect["label"] = "not-informative"
local_effect["label"] = local_effect.apply(lambda x: "misaligned" if x.informative and np.allclose(x.score, 0.0) else x.label, axis=1)
local_effect["label"] = local_effect.apply(lambda x: "aligned" if x.informative and np.allclose(x.score, 1.0) else x.label, axis=1)

In [115]:
label_mean = (
    local_effect.groupby(["agent_name", "q_type"])["label"]
          .value_counts(normalize=True)
          .rename("proportion")
          .reset_index()
)

label_mean = label_mean[label_mean.label != "aligned"]

In [116]:
label_mean.round(3)

,agent_name,q_type,label,proportion
1,20250603_Refact_Agent_claude-4-sonnet,local_effect,misaligned,0.398
2,20250603_Refact_Agent_claude-4-sonnet,local_effect,not-informative,0.112
4,20250720_Lingxi-v1.5_claude-4-sonnet-20250514,local_effect,misaligned,0.415
5,20250720_Lingxi-v1.5_claude-4-sonnet-20250514,local_effect,not-informative,0.093
7,20250805_openhands-Qwen3-Coder-480B-A35B-Instruct,local_effect,misaligned,0.400
8,20250805_openhands-Qwen3-Coder-480B-A35B-Instruct,local_effect,not-informative,0.078
9,20250807_mini-v1.7.0_gpt-5-mini,local_effect,misaligned,0.389
11,20250807_mini-v1.7.0_gpt-5-mini,local_effect,not-informative,0.241
13,20250928_trae_doubao_seed_code,local_effect,misaligned,0.394
14,20250928_trae_doubao_seed_code,local_effect,not-informative,0.149
